<a href="https://colab.research.google.com/github/ghalde194/APP1/blob/main/unificacao_Excel2_o_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import files
import pandas as pd
import numpy as np
from datetime import datetime
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
import warnings

# Silencia os avisos chatos do Python para deixar a tela limpa
warnings.filterwarnings('ignore', category=FutureWarning)

print("🚀 PROGRAMA - Unificação Premium e Inteligente (Datasul + Workday)")
print("=" * 90)

# =========================================================================
# 1. UPLOAD DO ARQUIVO
# =========================================================================
uploaded = files.upload()
nome_arquivo = next(iter(uploaded))

# =========================================================================
# 2. BUSCA FLEXÍVEL E LEITURA DAS ABAS
# =========================================================================
xls = pd.ExcelFile(nome_arquivo)
todas_as_abas = xls.sheet_names
print(f"Abas encontradas no seu arquivo: {todas_as_abas}")

aba_datasul = None
aba_workday = None
aba_kion = None
aba_dematic = None

# Procura as abas sem se importar com letras maiúsculas ou minúsculas
for aba in todas_as_abas:
    aba_lower = aba.strip().lower()
    if "datasul" in aba_lower:
        aba_datasul = aba
    elif "workday" in aba_lower:
        aba_workday = aba
    elif "kion" in aba_lower:
        aba_kion = aba
    elif "dematic" in aba_lower:
        aba_dematic = aba

if not aba_datasul or not aba_workday:
    print("\n❌ ERRO: Não encontrei as abas 'Datasul' ou 'Workday'.")
    raise Exception("Verifique os nomes das abas no seu arquivo Excel.")

print(f"✅ Usando aba Datasul: {aba_datasul}")
print(f"✅ Usando aba Workday: {aba_workday}")

# Leitura inicial das tabelas
df_datasul_principal = pd.read_excel(nome_arquivo, sheet_name=aba_datasul)
df_workday = pd.read_excel(nome_arquivo, sheet_name=aba_workday)

# =========================================================================
# 3. CONSOLIDAÇÃO DO DATASUL (Empilhando Kion e Dematic se existirem)
# =========================================================================
lista_dfs_datasul = [df_datasul_principal]

# Padronização de nomes de colunas que costumam vir bugados do sistema
mapa_colunas_hrp = {
    "Est": "Estabelecimento",
    "ata Admissã": "Data Admissão",
    "ata Admissão": "Data Admissão",
    "Cargo Básico-Des": "Cargo Básico",
    "Básico-Des": "Cargo Básico",
    "L Pgto-Desc": "Local Pgto-Descrição",
    "Descrição C": "Descrição Centro Custo",
    "Unid Negóci": "Unid Negócio"
}

if aba_kion:
    print(f"📦 Consolidando dados da aba extra: {aba_kion}")
    df_kion = pd.read_excel(nome_arquivo, sheet_name=aba_kion)
    df_kion.rename(columns=mapa_colunas_hrp, inplace=True)
    lista_dfs_datasul.append(df_kion)

if aba_dematic:
    print(f"📦 Consolidando dados da aba extra: {aba_dematic}")
    df_dematic = pd.read_excel(nome_arquivo, sheet_name=aba_dematic)
    df_dematic.rename(columns=mapa_colunas_hrp, inplace=True)
    lista_dfs_datasul.append(df_dematic)

# Junta tudo em um único panelão do Datasul
df_datasul = pd.concat(lista_dfs_datasul, ignore_index=True)

# =========================================================================
# 4. BUSCA INTELIGENTE DE COLUNAS (Evita que o código quebre)
# =========================================================================

# --- 🔍 Achar a Matrícula no Datasul ---
coluna_datasul_id = None
for col in df_datasul.columns:
    if "matrícula" in str(col).lower().strip() or "matricula" in str(col).lower().strip():
        coluna_datasul_id = col
        break
if not coluna_datasul_id:
    coluna_datasul_id = df_datasul.columns[0] # Se falhar, assume a 1ª coluna

# --- 🔍 Achar a Matrícula/ID no Workday (Aqui resolvemos o seu erro!) ---
coluna_workday_id = None
# Ordem de preferência de busca
palavras_chave_id = ["employee id", "payroll id", "id datasul", "datasul id", "matrícula", "matricula"]

for palavra in palavras_chave_id:
    for col in df_workday.columns:
        if palavra in str(col).lower().strip():
            coluna_workday_id = col
            break
    if coluna_workday_id:
        break

if not coluna_workday_id:
    coluna_workday_id = df_workday.columns[0] # Se falhar tudo, assume a 1ª coluna

# --- 🔍 Achar o Nome no Workday ---
coluna_nome_workday = None
palavras_chave_nome = ["legal name", "display format", "nome", "name"]
for palavra in palavras_chave_nome:
    for col in df_workday.columns:
        if palavra in str(col).lower().strip():
            coluna_nome_workday = col
            break
    if coluna_nome_workday:
        break

print(f"🔗 Cruzando Datasul ('{coluna_datasul_id}') ↔️ Workday ('{coluna_workday_id}')")

# =========================================================================
# 5. LIMPEZA E TRATAMENTO PREVENTIVO DE DUPLICADOS (A Solução que você pediu!)
# =========================================================================

# Padroniza as matrículas para número puro (evita erro de cruzamento texto vs número)
df_datasul["Matrícula_Limpa"] = pd.to_numeric(df_datasul[coluna_datasul_id], errors='coerce').astype('Int64')
df_workday["ID_Workday_Limpo"] = pd.to_numeric(df_workday[coluna_workday_id], errors='coerce').astype('Int64')

# Função para contar quantas palavras tem no nome e evitar nomes curtos incompletos
def tratar_e_contar_nome(df, col_nome):
    if col_nome and col_nome in df.columns:
        df["Nome_Aux"] = df[col_nome].astype(str).str.strip().str.title()
        df["Qtd_Palavras"] = df["Nome_Aux"].apply(lambda x: len(x.split()) if x not in ["Nan", "None", "-"] else 0)
    else:
        df["Nome_Aux"] = "-"
        df["Qtd_Palavras"] = 0
    return df

df_datasul = tratar_e_contar_nome(df_datasul, "Nome")

# Se o Workday estiver em inglês e o nome vier quebrado
if not coluna_nome_workday and "First Name" in df_workday.columns and "Last Name" in df_workday.columns:
    df_workday["Nome_Provisorio"] = df_workday["First Name"].astype(str) + " " + df_workday["Last Name"].astype(str)
    df_workday = tratar_e_contar_nome(df_workday, "Nome_Provisorio")
else:
    df_workday = tratar_e_contar_nome(df_workday, coluna_nome_workday)

# 🛡️ TRAVA ANTIDUPLICADOS: Ordena por quem tem MAIS palavras no nome e descarta o resto
df_datasul_valido = df_datasul.dropna(subset=["Matrícula_Limpa"]).sort_values(by="Qtd_Palavras", ascending=False)
df_datasul_valido = df_datasul_valido.drop_duplicates(subset=["Matrícula_Limpa"], keep='first')

df_workday_valido = df_workday.dropna(subset=["ID_Workday_Limpo"]).sort_values(by="Qtd_Palavras", ascending=False)
df_workday_valido = df_workday_valido.drop_duplicates(subset=["ID_Workday_Limpo"], keep='first')

# =========================================================================
# 6. CRUZAMENTO TOTAL (Full Outer Merge)
# =========================================================================
df_final = df_datasul_valido.merge(
    df_workday_valido,
    left_on="Matrícula_Limpa",
    right_on="ID_Workday_Limpo",
    how="outer",
    suffixes=("", "_Workday")
)

df_final["Matrícula"] = df_final["Matrícula_Limpa"].fillna(df_final["ID_Workday_Limpo"])

# Escolha do melhor nome disponível
df_final["Nome_F"] = df_final["Nome_Aux"].fillna(df_final["Nome_Aux_Workday"])
mascara_inverter = df_final["Qtd_Palavras_Workday"].fillna(0) > df_final["Qtd_Palavras"].fillna(0)
df_final.loc[mascara_inverter, "Nome_F"] = df_final.loc[mascara_inverter, "Nome_Aux_Workday"]

df_final["Nome"] = df_final["Nome_F"].astype(str).str.strip().str.title()
df_final.loc[df_final["Nome"].isin(["Nan", "None", "Nat", "", "-"]), "Nome"] = "-"

# =========================================================================
# 7. ORGANIZAÇÃO DE COLUNAS E BASES DE AUDITORIA
# =========================================================================
colunas_organizadas = [
    "Matrícula", "Nome", "Sexo", "Situação", "Estabelecimento",
    "Unid Negócio", "Cargo Básico", "Local Pgto-Descrição", "Data Admissão",
    "Employee Type", "Position", "Location", "Company"
]
colunas_finais = [col for col in colunas_organizadas if col in df_final.columns]
df_base_unificada = df_final[colunas_finais].copy()

# Criação das abas de auditoria de divergência
matriculas_datasul_limpas = df_datasul_valido["Matrícula_Limpa"].dropna().unique()
matriculas_workday_limpas = df_workday_valido["ID_Workday_Limpo"].dropna().unique()

df_so_datasul = df_datasul_valido[~df_datasul_valido["Matrícula_Limpa"].isin(matriculas_workday_limpas)].copy()
df_so_workday = df_workday_valido[~df_workday_valido["ID_Workday_Limpo"].isin(matriculas_datasul_limpas)].copy()

df_so_datasul["Nome"] = df_so_datasul["Nome_Aux"]
df_so_workday["Nome"] = df_so_workday["Nome_Aux"]

df_so_datasul.drop(columns=["Matrícula_Limpa", "Nome_Aux", "Qtd_Palavras"], inplace=True, errors='ignore')
df_so_workday.drop(columns=["ID_Workday_Limpo", "Nome_Aux", "Qtd_Palavras", "Nome_Provisorio"], inplace=True, errors='ignore')

bases = {"Base Unificada": df_base_unificada, "Apenas Datasul": df_so_datasul, "Apenas Workday": df_so_workday}

# =========================================================================
# 8. LIMPEZA FINAL DE NULOS E PADRONIZAÇÃO DE DATAS
# =========================================================================
for nome_aba, df_temp in bases.items():
    # Identifica colunas de data e as coloca no padrão brasileiro
    colunas_data = [c for c in df_temp.columns if "data" in str(c).lower() or "date" in str(c).lower()]
    for col_dt in colunas_data:
        df_temp[col_dt] = pd.to_datetime(df_temp[col_dt], errors='coerce')
        df_temp[col_dt] = df_temp[col_dt].dt.strftime('%d/%m/%Y').fillna("-")

    df_temp.fillna("-", inplace=True)
    df_temp.replace([np.nan, "nan", "None", "NaT", ""], "-", inplace=True)

    if "Nome" in df_temp.columns:
        # Tira linhas fantasmas do Excel que vem zeradas
        linhas_vazias = (df_temp["Nome"] == "-") & (df_temp.drop(columns=["Nome"]).eq("-").all(axis=1))
        df_temp = df_temp[~linhas_vazias]
        df_temp.sort_values(by='Nome', ascending=True, inplace=True)

    df_temp.reset_index(drop=True, inplace=True)
    bases[nome_aba] = df_temp

# =========================================================================
# 9. SALVAR NO EXCEL COM FORMATAÇÃO PREMIUM
# =========================================================================
nome_saida = f"Planilha_Unificada_Premium_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"

with pd.ExcelWriter(nome_saida, engine="openpyxl") as writer:
    for nome_aba, df_temp in bases.items():
        df_temp.to_excel(writer, sheet_name=nome_aba, index=False)
        ws = writer.sheets[nome_aba]
        ws.views.sheetView[0].showGridLines = True

        # Paleta de cores e fontes
        cor_cabecalho = PatternFill(start_color="1A365D", end_color="1A365D", fill_type="solid")
        cor_zebra = PatternFill(start_color="F8FAFC", end_color="F8FAFC", fill_type="solid")
        fonte_cabecalho = Font(name="Segoe UI", size=11, bold=True, color="FFFFFF")
        fonte_corpo = Font(name="Segoe UI", size=10)
        alinhar_centro = Alignment(horizontal="center", vertical="center")
        alinhar_esquerda = Alignment(horizontal="left", vertical="center")
        borda_fina = Side(border_style="thin", color="E2E8F0")
        borda_celula = Border(left=borda_fina, right=borda_fina, top=borda_fina, bottom=borda_fina)

        # Estilização do cabeçalho
        for cell in ws[1]:
            cell.fill = cor_cabecalho
            cell.font = fonte_cabecalho
            cell.alignment = alinhar_centro
            cell.border = borda_celula

        # Estilização do corpo (Zebrado e Alinhamentos)
        for r_idx, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row, min_col=1, max_col=ws.max_column), start=2):
            for cell in row:
                cell.font = fonte_corpo
                cell.border = borda_celula
                cell.alignment = alinhar_esquerda

                if cell.value == "-":
                    cell.alignment = alinhar_centro

                nome_coluna = ws.cell(row=1, column=cell.column).value
                # Força matrícula a ser texto para o Excel não tirar o zero à esquerda
                if any(termo in str(nome_coluna).lower() for termo in ["matrícula", "id", "employee"]):
                    cell.number_format = "@"
                if any(term in str(nome_coluna).lower() for term in ["matrícula", "date", "data", "id", "sexo", "status", "situação"]):
                    cell.alignment = alinhar_centro
                if r_idx % 2 == 0:
                    cell.fill = cor_zebra

        # Autoajuste automático de largura de colunas
        for col in ws.columns:
            column_letter = col[0].column_letter
            max_len_conteudo = max(len(str(cell.value or '')) for cell in col)
            ws.column_dimensions[column_letter].width = min(max_len_conteudo + 4, 45)

        ws.freeze_panes = "A2"

print(f"\n✅ Arquivo gerado com sucesso: {nome_saida}")
files.download(nome_saida)

🚀 PROGRAMA - Unificação Premium e Inteligente (Datasul + Workday)


Saving Reportal Function Março26 - Leonardo Homework.xlsx to Reportal Function Março26 - Leonardo Homework (21).xlsx
Abas encontradas no seu arquivo: ['Datasul', 'Workday', 'HRP Kion', 'HRP Dematic']
✅ Usando aba Datasul: Datasul
✅ Usando aba Workday: Workday
📦 Consolidando dados da aba extra: HRP Kion
📦 Consolidando dados da aba extra: HRP Dematic
🔗 Cruzando Datasul ('Matrícula') ↔️ Workday ('Employee ID')

✅ Arquivo gerado com sucesso: Planilha_Unificada_Premium_20260402_1231.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>